Setting up DLT(Delta Live Table) Scenario Dataset
- setting up datasets and objects for DLT(Delta Live Table).

For working with DLT(Delta Live Table) Pipeline, we need to go with multi-node cluster 
 and also access mode :shared

compute ---> creating multi-node cluster

General
Compute name :Neeraj cluster

Policy :Shared Compute

Performance

Machine learning
Databricks runtime :16.4 LTS (Scala 2.12)
                    Scala 2.12, Spark 3.5.2

Photon acceleration --> disable

Worker type :Standard_DS3_v2 8 GB Memory, 4 Cores
Min :1   Max:1

-> CREATE COMPUTE

In [0]:
#spark.sql("CREATE CATALOG IF NOT EXISTS dev")
#spark.sql("CREATE DATABASE IF NOT EXISTS dev.demodb1")

Goto Datalake -> Create one folder in dbfscontainer -> Add directory -> dataset_ch9
and upload 7files into this

creating External Location

In [0]:

#CREATE EXTERNAL LOCATION IF NOT EXISTS `external-data`
#URL  'abfss://dbfscontainer@mystoragelakeadb6pmgroup.dfs.core.windows.net/dataset_dlt'
#WITH (CREDENTIAL `adb6pm-storage-credential`);

Creating external volume

In [0]:

#CREATE EXTERNAL VOLUME IF NOT EXISTS dev.demodb1.landing_zone
#LOCATION 'abfss://dbfscontainer@mystoragelakeadb6pmgroup.dfs.core.windows.net/dataset_dlt'

Create 2 directories under landing_zone.
1. customers
2. invoices

once you create, it will be visible at ADLS and Landing_zone (both places).

Either way you create - by using below code or manually.

In [0]:
#%fs
#mkdirs /Volumes/dev/demodb/landing_zone/customers/

#%fs
#mkdirs /Volumes/dev/demodb/landing_zone/invoices/

In [0]:
#%fs
#ls /Volumes/dev/demodb1/landing_zone

In [0]:
#%fs
#cp /Volumes/dev/demodb1/landing_zone/customers_1.csv  /Volumes/dev/demodb1/landing_zone/customers

Copy 2021 invoice file.

In [0]:
#%fs
#cp /Volumes/dev/demodb1/landing_zone/invoices_2021.csv  /Volumes/dev/demodb1/landing_zone/invoices

Copy 2022 invoice file.

In [0]:
#%fs
#cp /Volumes/dev/demodb1/landing_zone/invoices_2022.csv  /Volumes/dev/demodb1/landing_zone/invoices

Create STREAMING TABLE customers_raw.

Bronze layer: 

Create bronze table by loading data from landing zone.

first from landing zone ingest data into 2 tables

customers_raw
invoices_raw
we want to do it using auto-loader.

I want DLT pipeline only to create table and load data.

DLT pipeline code can be written in 2 ways

- using sparksql

- using python

%md
1. Using Python:

Import dlt function.

Create your bronze layer tables and ingesting data from landing zone.

here we want DLT (Delta Live Table) to create and fill table using incremental approach (Streaming).

here we want to create a streaming table and
next we tell DLT (Delta Live Table) from where to get the data from 

get from cloudfiles() function

- cloudfiles() is our auto loader

in cloudfiles() function, we provide parameters to define autoloader configurations
and landingzone directory

1st paramter is landingzone directory from where cloudfiles() should start
ingesting data

ex: cloud_files('Volumes/dev/demodb/landing_zone/customers'

2nd parameter--->cloud file format

3rd paramter---->autoloader configuration.

by default it comes with default configuration

and other additional configurations we can provide.

        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/Volumes/dev/demodb1/landing_zone/customers")

In [0]:
import dlt
from pyspark.sql.functions import current_timestamp

@dlt.table(name="customers_raw")
def customers_raw():
    return (
        spark.readStream
            .format("cloudFiles") # Format  
            .option("cloudFiles.format", "csv") # File format
            .option("cloudFiles.inferColumnTypes", "true") # Infer schema - configuration
            .load("/Volumes/dev/demodb1/landing_zone/customers") # Load data.
            .withColumn("load_time", current_timestamp())
    )

In [0]:
%python

# spark.sql("CREATE OR REFRESH STREAMING TABLE customers_raw AS SELECT *, current_timestamp() as load_time from cloud_files('/Volumes/dev/demodb1/landing_zone/customers','csv',map('cloudFiles.inferColumnTypes','true')) ")

STREAMING TABLE invoices_raw.

Bronze layer: Invoices_Raw. 

Create bronze table by loading data from landing zone.

In [0]:
import dlt
from pyspark.sql.functions import current_timestamp

@dlt.table(name="invoices_raw")
def invoices_raw():
    return (
        spark.readStream
            .format("cloudFiles") # Format
            .option("cloudFiles.format", "csv") # File format
            .option("cloudFiles.inferColumnTypes", "true") # Infer schema
            .load("/Volumes/dev/demodb1/landing_zone/invoices") # Load data
            .withColumn("load_time", current_timestamp())
    )

In [0]:
%python

#spark.sql("CREATE OR REFRESH STREAMING TABLE invoices_raw AS SELECT *,current_timestamp() as load_time from cloud_files('/Volumes/dev/demodb1/landing_zone/invoices','csv',map('cloudFiles.inferColumnTypes','true')) ")

In [0]:
%python
#spark.sql("CREATE OR REFRESH STREAMING TABLE customers_cleaned(CONSTRAINT valid_customer EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW) AS SELECT CustomerID as customer_id, CustomerName as customer_name, load_time from STREAM(live.customers_raw)")

Silver layer: Invoices_Cleaned table.

In [0]:
import dlt
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    month
)

@dlt.table(
    name="invoices_cleaned",
    comment="Cleaned invoices data with valid invoice and quantity",
    partition_cols=["invoice_year", "country"]
)
@dlt.expect_or_drop(
    "valid_invoice_and_qty",
    "invoice_no IS NOT NULL AND quantity > 0"
)
def invoices_cleaned():
    return (
        dlt.read_stream("invoices_raw")
            .select(
                col("InvoiceNo").alias("invoice_no"),
                col("StockCode").alias("stock_code"),
                col("Description").alias("description"),
                col("Quantity").alias("quantity"),
                to_date(col("InvoiceDate"), "d-M-y H.m").alias("invoice_date"),
                col("UnitPrice").alias("unit_price"),
                col("CustomerID").alias("customer_id"),
                col("Country").alias("country"),
                year(to_date(col("InvoiceDate"), "d-M-y H.m")).alias("invoice_year"),
                month(to_date(col("InvoiceDate"), "d-M-y H.m")).alias("invoice_month"),
                col("load_time")
            )
    )

In [0]:
%python

#spark.sql("CREATE OR REFRESH STREAMING TABLE invoices_cleaned (CONSTRAINT valid_invoice_and_qty EXPECT (invoice_no IS NOT NULL AND quantity >0) ON VIOLATION DROP ROW) PARTITIONED BY (invoice_year,country) AS SELECT InvoiceNo as invoice_no, StockCode as stock_code, Description	as description, Quantity as quantity, to_date(InvoiceDate,\"d-M-y H.m\") as invoice_date, UnitPrice	as unit_price, CustomerID	as customer_id, Country	as country, year(to_date(InvoiceDate,\"d-M-y H.m\")) as invoice_year, month(to_date(InvoiceDate,\"d-M-y H.m\")) as invoice_month, load_time FROM STREAM(live.invoices_raw)")

Silver layer: Customers_Cleaned and Customers tables.

In [0]:
# 1️⃣ Source cleaned table
@dlt.table(
    name="customers_cleaned"
)
@dlt.expect_or_drop("valid_customer", "customer_id IS NOT NULL")
def customers_cleaned():
    return (
        dlt.read_stream("customers_raw")
            .selectExpr(
                "CustomerID as customer_id",
                "CustomerName as customer_name",
                "load_time"
            )
    )
dlt.create_streaming_table(
    name="customers"
)
# 2️⃣ Target table (SCD Type 2) using apply_changes
dlt.apply_changes(
    target="customers",  # must match catalog + database
    source="customers_cleaned",       # the function above
    keys=["customer_id"],
    sequence_by="load_time",
    stored_as_scd_type=2              # for historical tracking
)

Silver layer: Invoices table

In [0]:
import dlt

# Create or refresh the streaming target table
dlt.create_streaming_table(
    name="invoices"
)

# Apply CDC changes (SCD Type 1)
dlt.apply_changes(
    target="invoices",
    source="invoices_cleaned",
    keys=["invoice_no", "stock_code", "invoice_date"],
    sequence_by="load_time",

    # CDC options
    stored_as_scd_type=1
)

Create output table: Daily Sales.
Gold layer: Daily_Sales

In [0]:
import dlt
from pyspark.sql.functions import col, sum, round

@dlt.table(
    name="daily_sales",
    comment="Daily sales for United Kingdom in 2022"
)
def daily_sales():
    return (
        dlt.read("invoices")   # use dlt.read(), NOT read_stream for aggregates
            .filter(
                (col("invoice_year") == 2022) &
                (col("country") == "United Kingdom")
            )
            .groupBy(
                "country",
                "invoice_year",
                "invoice_month",
                "invoice_date"
            )
            .agg(
                round(sum(col("quantity") * col("unit_price")), 2)
                    .alias("total_sales")
            )
    )